# Narcolepsy Discriminative Model

This is example code of how to discriminate narcoleptic patients using electronic health record data.  
EHR data needed:
- Clinical visit notes
- ICD codes
- Medications

In [ ]:
import polars as pl
import pandas as pd
from narcolepsy_model import NarcolepsyModel
from pathlib import Path

model = NarcolepsyModel(
    model_type='nt1_vs_not' # or 'nt2_vs_not' or 'nt12_vs_not', this is preloading the trained model
)

data_path = Path('data/discriminative-modeling') # path to data directory

data = {
    'note' : pl.read_parquet(data_path / 'notes.parquet'),
    'icd' : pl.read_parquet(data_path / 'icd.parquet'),
    'med' : pl.read_parquet(data_path / 'med.parquet')
}


# can use pandas dataframes as well
# output will be polars dataframes, but easy to convert back to pandas if needed

# data = {
#     'note' : pd.read_parquet(data_path / 'note.parquet')
#     'icd' : pd.read_parquet(data_path / 'icd.parquet')
#     'med' : pd.read_parquet(data_path / 'med.parquet')
# }

In [ ]:
# this is the schema used by by the feature extraction pipeline
# ensure your data matches this schema

# id : patient identifier (string)
# icd : ICD 9 or 10 diagnostic code (string)
# med : medication (string)
# note : clinical visit note (string)
# date : date of respective event (date -- please ensure this is date, not datetime)

model.config['schema']

{'icd': {'id': String, 'icd': String, 'date': Date},
 'med': {'id': String, 'med': String, 'date': Date},
 'note': {'id': String, 'note': String, 'date': Date}}

In [ ]:
feat, pred = model.run(
    data,
    show_progress=True, # loading bar
    return_features=True # returns the features extracted from the notes
)

2026-03-02 13:10:47,078	INFO worker.py:1841 -- Started a local Ray instance.
Processing narcolepsy notes: 100%|██████████| 8990/8990 [00:28<00:00, 310.85it/s]


In [ ]:
feat.write_parquet(data_path / 'features.parquet')
feat.head()

id,date,^347\.?[0|1]1|^G47\.?4[1|2]1,^347\.?[0|1]0|^G47\.?4[1|2]9,^G47\.?1|^780\.?5[3|4],adderall,ambien,clomipramine,concerta,cymbalta,dexedrine,dextroamphetamine,duloxetine,effexor,fluoxetine,imipramine,modafinil,nuvigil,paroxetine,pitolisant,protriptyline,provigil,prozac,ritalin,sertraline,sodium oxybate,solriamfetol,sunosi,venlafaxine,wakix,xyrem,xywav,50 improv_,abnorm_,abnorm find_,abnorm glucos_,abnorm weight_,…,type_neg_,type walk_neg_,unrefresh_neg_,unrefresh sleep_neg_,use_neg_,venlafaxin_neg_,visit_neg_,vivid_neg_,vivid dream_neg_,wake_neg_,wake test_neg_,walk_neg_,walk 30_neg_,walk activ_neg_,walk q_neg_,watch_neg_,watch diet_neg_,weak_neg_,wean_neg_,week_neg_,weight_neg_,weight decreas_neg_,weight discuss_neg_,weight gain_neg_,weight increas_neg_,weight loss_neg_,weight mainten_neg_,weight manag_neg_,work_neg_,work diet_neg_,work routin_neg_,worri_neg_,xr_neg_,xyrem_neg_,yes_neg_,zanaflex_neg_,zoloft_neg_
str,date,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""150041900""",2017-09-27,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,…,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""150012174""",2019-03-04,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
"""179002121""",2022-12-25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""179017431""",2021-11-19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""177519609""",2017-02-27,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,1,0,1,1,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [ ]:
# this is the returned predictions from the model for each id / date in the input data
# prediction = 1 is based on an unoptimized default threshold of prob_YES > 0.5

pred.head()

id,date,prob_NO,prob_YES,prediction
str,date,f64,f64,i32
"""150041900""",2017-09-27,0.938451,0.061549,0
"""150012174""",2019-03-04,0.909572,0.090428,0
"""179002121""",2022-12-25,0.999112,0.000888,0
"""179017431""",2021-11-19,0.999112,0.000888,0
"""177519609""",2017-02-27,0.9985,0.0015,0


In [ ]:
# can convert polars to pandas with .to_pandas()

pred.to_pandas().head()

,id,date,prob_NO,prob_YES,prediction
0,150041900,2017-09-27,0.938451,0.061549,0
1,150012174,2019-03-04,0.909572,0.090428,0
2,179002121,2022-12-25,0.999112,0.000888,0
3,179017431,2021-11-19,0.999112,0.000888,0
4,177519609,2017-02-27,0.998500,0.001500,0


## model training

The code below is provided for reproducibility.
Annotation labels are provided in the 'notes.parquet' file.  
Annotation scheme is
1. NT1
2. NT2 / IH
3. Unsure (remove from training)
4. Not narcolepsy

See cell below for example code.  
To train NT1 vs Others, set annotation label 1 as 1, the others as 0.  
To train NT2IH vs Others, set annotation label 2 as 1, the others as 0.  

In [3]:
from model_comp import leave_one_source_out_validation, define_models_config, plot_leave_one_out_curves, train_final_production_model
import ast
from pathlib import Path
import polars as pl

data_path = Path('data')

note = pl.read_parquet('/home/niels/Desktop/pheno_models/narcolepsy/data/discriminative-modeling/notes.parquet')
feat = pl.read_parquet('/home/niels/Desktop/pheno_models/narcolepsy/data/discriminative-modeling/features.parquet')

# feature matrix and labels for evaluation
X = note.select('annot', 'cohort').hstack(feat).drop('id', 'date').with_columns(
    pl.col('annot').replace({
        # for this example, we will do NT1 vs Others training
        # 1:1,
        # 2:0,
        # 3:None, # remove unsure cases from training
        # 4:0

        # to train NT2 / IH vs Others, use this; the code below remains the same (aside from saving results in a different directory)
        # 1:0,
        # 2:1,
        # 3:None, # remove unsure cases from training
        # 4:0

        # to train NT1 / NT2 / IH vs Others, use this; the code below remains the same (aside from saving results in a different directory)
        1:1,
        2:1,
        3:None, # remove unsure cases from training
        4:0
    })
)
X = X.drop_nulls()
X.head()

annot,cohort,^347\.?[0|1]1|^G47\.?4[1|2]1,^347\.?[0|1]0|^G47\.?4[1|2]9,^G47\.?1|^780\.?5[3|4],adderall,ambien,clomipramine,concerta,cymbalta,dexedrine,dextroamphetamine,duloxetine,effexor,fluoxetine,imipramine,modafinil,nuvigil,paroxetine,pitolisant,protriptyline,provigil,prozac,ritalin,sertraline,sodium oxybate,solriamfetol,sunosi,venlafaxine,wakix,xyrem,xywav,50 improv_,abnorm_,abnorm find_,abnorm glucos_,abnorm weight_,…,type_neg_,type walk_neg_,unrefresh_neg_,unrefresh sleep_neg_,use_neg_,venlafaxin_neg_,visit_neg_,vivid_neg_,vivid dream_neg_,wake_neg_,wake test_neg_,walk_neg_,walk 30_neg_,walk activ_neg_,walk q_neg_,watch_neg_,watch diet_neg_,weak_neg_,wean_neg_,week_neg_,weight_neg_,weight decreas_neg_,weight discuss_neg_,weight gain_neg_,weight increas_neg_,weight loss_neg_,weight mainten_neg_,weight manag_neg_,work_neg_,work diet_neg_,work routin_neg_,worri_neg_,xr_neg_,xyrem_neg_,yes_neg_,zanaflex_neg_,zoloft_neg_
i64,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,"""bidmc""",0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,…,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,"""bidmc""",0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
0,"""emory""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,"""emory""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,"""stan""",0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,1,0,1,1,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [4]:
models_config = define_models_config()
models_config.keys()

# you can drop any models you don't want to test
# models_config.pop('XGBoost')

dict_keys(['LogisticRegression', 'RandomForest', 'GradientBoosting', 'XGBoost'])

In [5]:
# grid search, leave one cohort out validation
results, curves_data = leave_one_source_out_validation(
    X,
    source_col="cohort", # column name for site cross validation
    target_col="annot", # column name for target variable
    models_config={'LogisticRegression':models_config['LogisticRegression']}, # will train all models
    scoring_metric="average_precision", # can be any metric from sklearn.metrics for optimizing hyperparameters
    output_dir="results/nt1_vs_others/", # directory to save results, CHANGE for nt2ih_vs_not
    random_state=42,
    save_fold_models=True # save individual cv folds
)

results.write_csv('results/results.csv')

plot_leave_one_out_curves(
    curves_data,
    results,
    output_dir='results'
)

Leave-one-source-out validation for binary classification with 2 classes: [0, 1]
Found 5 unique sources: ['bidmc', 'bch', 'emory', 'mgb', 'stan']

===== Evaluating LogisticRegression =====
Detected 924 binary features and 0 continuous features

Training LogisticRegression on all sources except: bidmc
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best parameters for LogisticRegression excluding bidmc: {'model__C': 0.1, 'model__class_weight': None, 'model__l1_ratio': 0.25, 'model__max_iter': 5000, 'model__random_state': 42, 'model__solver': 'saga'}
Saving fold model to: results/nt1_vs_others/fold_models_LogisticRegression/LogisticRegression_train_all_except_bidmc_test_bidmc.pkl
Saved fold model info to: results/nt1_vs_others/fold_models_LogisticRegression/LogisticRegression_train_all_except_bidmc_test_bidmc_info.json
Testing LogisticRegression on held-out source: bidmc
Confusion matrix saved to results/nt1_vs_others/confusion_matrices_LogisticRegression/confusion_matrix_L

### train final model (using all of the data)

In [ ]:
# the metrics above are from cross-validation
# can use best hyperparameters from model of choice to train final model

ast.literal_eval(results.filter(pl.col('model') == 'RandomForest').sort('f1', descending=True)[0, 'best_params'])

{'model__class_weight': None,
 'model__max_depth': None,
 'model__min_samples_split': 10,
 'model__n_estimators': 200,
 'model__random_state': 42}

In [ ]:
final_model, model_info = train_final_production_model(
    X,
    source_col="cohort",
    target_col="annot",
    best_model_config=(
        'RandomForest',
        models_config['RandomForest'][0],
        ast.literal_eval(results.filter(pl.col('model') == 'RandomForest').sort('f1', descending=True)[0, 'best_params'])
    ),
    output_dir="results/nt1_vs_others/final_model/", # directory to save final model, CHANGE for nt2ih_vs_not
    random_state=42
)

## extract features for predictive modeling

In [ ]:
import polars as pl
import pandas as pd
from narcolepsy_model import NarcolepsyModel
from pathlib import Path

model = NarcolepsyModel()

data_path = Path('data/predictive-modeling') # path to data directory PREDICTIVE

data = {
    'note' : pl.read_parquet(data_path / 'notes.parquet'), # this is the larger dataset of notes used for prediction modeling
    'icd' : pl.read_parquet(data_path / 'icd.parquet'),
    'med' : pl.read_parquet(data_path / 'med.parquet')
}

feat = model.preprocess( # preprocess only extracts features, does not return predictions
    data,
    show_progress=True, # loading bar
)

feat.write_parquet(data_path / 'features.parquet')